In [19]:
!pip install torch_optimizer
!pip install medpy
!pip install albumentations

In [20]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
from PIL import Image

from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.patches as mpatches

In [21]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torchvision.transforms.functional as TF
import random


class SynapseDataset(Dataset):
    def __init__(self, npz_files, augment=False, image_size=(112, 112)):
        self.files = []

        for f in npz_files:
            npz = np.load(f)
            image, label = npz['image'], npz['label']

            if np.max(image) > 0 and np.max(label) > 0:
                self.files.append(f)

        self.augment = augment
        self.image_size = image_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        npz = np.load(self.files[idx])

        image = npz['image'].astype(np.float32)
        label = npz['label'].astype(np.int64)

        # Min-max normalization
        image = (image - image.min()) / (
            image.max() - image.min() + 1e-8
        )

        image = np.expand_dims(image, axis=0)

        image = torch.from_numpy(image)
        label = torch.from_numpy(label)

        # Resize
        image = TF.resize(image, self.image_size)

        label = TF.resize(
            label.unsqueeze(0).float(),
            self.image_size,
            interpolation=TF.InterpolationMode.NEAREST
        ).squeeze(0).long()

        # Slightly stronger augmentation
        if self.augment:
            image, label = self.random_augment(image, label)

        return image, label

    def random_augment(self, image, label):

        # Horizontal flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            label = TF.hflip(label)

        # Slightly larger rotation
        angle = random.uniform(-10, 10)

        image = TF.rotate(
            image,
            angle,
            interpolation=TF.InterpolationMode.BILINEAR
        )

        label = TF.rotate(
            label.unsqueeze(0),
            angle,
            interpolation=TF.InterpolationMode.NEAREST
        ).squeeze(0)

        # Slight brightness variation
        if random.random() > 0.5:
            factor = random.uniform(0.90, 1.10)
            image = TF.adjust_brightness(image, factor)

        # Slight contrast variation
        if random.random() > 0.5:
            factor = random.uniform(0.90, 1.10)
            image = TF.adjust_contrast(image, factor)

        return image, label

In [22]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

dataset_path = "/kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz"
all_npz_files = sorted([os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith('.npz')])

# Split
train_files, val_files = train_test_split(all_npz_files, test_size=0.25, random_state=42)

train_dataset = SynapseDataset(train_files)
val_dataset = SynapseDataset(val_files)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4)

In [23]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape) 
print("Mask batch shape:", masks.shape)   

Image batch shape: torch.Size([4, 1, 112, 112])
Mask batch shape: torch.Size([4, 112, 112])


In [24]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class ViTBlock(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ViTEncoder(nn.Module):
    def __init__(self, dim=768, num_layers=12):
        super().__init__()
        self.layers = nn.ModuleList([ViTBlock(dim) for _ in range(num_layers)])

    def forward(self, x):
        B, C, H, W = x.shape
        seq = x.flatten(2).transpose(1, 2)  
        feats = []
        for i, blk in enumerate(self.layers):
            seq = blk(seq)
            if i in [2, 5, 8, 11]:
                tmp = seq.transpose(1, 2).reshape(B, C, H, W)
                feats.append(tmp)
        return feats  

class ResSkip(nn.Module):
    def __init__(self, res_ch, out_ch, res_up=1):
        super().__init__()
        self.res_conv = nn.Conv2d(res_ch, out_ch, 3, padding=1)
        self.res_deconvs = nn.ModuleList([nn.ConvTranspose2d(out_ch, out_ch, 2, 2) for _ in range(res_up)])

    def forward(self, res_feat):
        r = self.res_conv(res_feat)
        for de in self.res_deconvs:
            r = de(r)
        return r


class BottleneckFusion(nn.Module):
    def __init__(self, res_in_ch=2048, vit_in_ch=768, out_ch=768, num_heads=8):
        super().__init__()
        self.res_up = nn.ConvTranspose2d(res_in_ch, res_in_ch, kernel_size=2, stride=2)
        self.attn_norm = nn.LayerNorm(res_in_ch)
        self.attn = nn.MultiheadAttention(res_in_ch, num_heads=num_heads, batch_first=True)
        self.vit_proj = nn.Linear(vit_in_ch, res_in_ch)
        self.conv = nn.Conv2d(res_in_ch + vit_in_ch, out_ch, 3, padding=1)

    def forward(self, res4, vit12):
        r = self.res_up(res4)

        min_h = min(r.shape[2], vit12.shape[2])
        min_w = min(r.shape[3], vit12.shape[3])
        r = r[:, :, :min_h, :min_w]
        vit12 = vit12[:, :, :min_h, :min_w]

        B, C_r, H, W = r.shape
        r_seq = r.flatten(2).transpose(1, 2)        # (B, N, C_r)
        vit_seq = vit12.flatten(2).transpose(1, 2)  # (B, N, C_v)

        vit_seq_proj = self.vit_proj(vit_seq)    

        r_seq_norm = self.attn_norm(r_seq)
        attn_out, _ = self.attn(r_seq_norm, vit_seq_proj, vit_seq_proj)
        r_seq = r_seq + attn_out

        r = r_seq.transpose(1, 2).reshape(B, C_r, H, W)

        x = torch.cat([r, vit12], dim=1)
        x = self.conv(x)
        return x

class SegmentationModel(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)

        old_conv1 = backbone.conv1
        backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv1.out_channels,
            kernel_size=old_conv1.kernel_size,
            stride=old_conv1.stride,
            padding=old_conv1.padding,
            bias=False
        )
        with torch.no_grad():
            backbone.conv1.weight[:] = old_conv1.weight.mean(dim=1, keepdim=True)

        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # 256 ch
        self.layer2 = backbone.layer2   # 512 ch
        self.layer3 = backbone.layer3   # 1024 ch
        self.layer4 = backbone.layer4   # 2048 ch

        self.l3_to_vit = nn.Sequential(
            nn.Conv2d(1024, 768, 3, padding=1),
            nn.Conv2d(768, 768, 3, padding=1)
        )

        self.vit = ViTEncoder(dim=768, num_layers=12)
        self.bottleneck = BottleneckFusion(res_in_ch=2048, vit_in_ch=768, out_ch=768)

        # Decoder
        self.up1 = nn.ConvTranspose2d(768, 512, 2, 2)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.up4 = nn.ConvTranspose2d(128,  64, 2, 2)

        # Skip connections (ablation: ResNet-only, no HDA/CBAM fusion)
        self.skip1 = ResSkip(res_ch=1024, out_ch=512, res_up=1)
        self.skip2 = ResSkip(res_ch=512,  out_ch=256, res_up=1)
        self.skip3 = ResSkip(res_ch=256,  out_ch=128, res_up=1)

        # Input skip 
        self.input_skip = nn.Conv2d(1, 64, 3, padding=1)

        self.final_conv = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        l0 = self.stem(x)      
        l1 = self.layer1(l0)   
        l2 = self.layer2(l1)   
        l3 = self.layer3(l2)     
        l4 = self.layer4(l3)   

        vit_in = self.l3_to_vit(l3)                      
        vit_feats = self.vit(vit_in)
        vit_L12 = vit_feats[-1]

        b = self.bottleneck(l4, vit_L12)                

        up1 = self.up1(b)   + self.skip1(l3)     
        up2 = self.up2(up1) + self.skip2(l2)      
        up3 = self.up3(up2) + self.skip3(l1)     
        up4 = self.up4(up3) + self.input_skip(x)          

        out = self.final_conv(up4)                        
        return out

In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SegmentationModel(num_classes=9).to(device)

In [26]:
import torch
import torch.nn.functional as F
import numpy as np
from medpy.metric.binary import hd

def dice_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    intersection = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    union = preds_onehot.sum(dim=(0,2,3)) + targets_onehot.sum(dim=(0,2,3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean().item()

def iou_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    intersection = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    union = (preds_onehot + targets_onehot - preds_onehot*targets_onehot).sum(dim=(0,2,3))
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

def pixel_accuracy(preds, targets):
    return (preds.argmax(dim=1) == targets).float().mean().item()

def precision_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    tp = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    fp = (preds_onehot * (1 - targets_onehot)).sum(dim=(0,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    return precision.mean().item()

def recall_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    tp = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    fn = ((1 - preds_onehot) * targets_onehot).sum(dim=(0,2,3))
    recall = (tp + eps) / (tp + fn + eps)
    return recall.mean().item()

def f1_score(preds, targets, eps=1e-7):
    prec = precision_score(preds, targets, eps)
    rec = recall_score(preds, targets, eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    return f1

def hausdorff_distance(preds, targets):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    distances = []
    for b in range(preds_onehot.shape[0]):
        for c in range(preds_onehot.shape[1]):
            p = preds_onehot[b,c].cpu().numpy().astype(bool)
            t = targets_onehot[b,c].cpu().numpy().astype(bool)
            if p.sum() == 0 or t.sum() == 0:
                distances.append(np.nan)
            else:
                try:
                    distances.append(hd(p, t))
                except:
                    distances.append(np.nan)
    return np.nanmean(distances)


In [29]:
import torch
import torch.nn as nn
import torch.optim as optim

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, preds, targets):
        num_classes = preds.shape[1]
        preds = F.softmax(preds, dim=1)
        preds_one_hot = F.one_hot(preds.argmax(dim=1), num_classes=num_classes).permute(0,3,1,2).float()
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0,3,1,2).float()

        intersection = (preds_one_hot * targets_one_hot).sum(dim=(0,2,3))
        union = preds_one_hot.sum(dim=(0,2,3)) + targets_one_hot.sum(dim=(0,2,3))
        dice = (2. * intersection + self.eps) / (union + self.eps)
        return 1 - dice.mean()

ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

def combined_loss(outputs, targets, ce_weight=0.5, dice_weight=0.5):
    return ce_weight * ce_loss(outputs, targets) + dice_weight * dice_loss(outputs, targets)

criterion = combined_loss
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5,weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=4, factor=0.5)

In [30]:
from torch.amp import autocast, GradScaler
import torch
import torch.nn.functional as F
from tqdm import tqdm

scaler = GradScaler() 
accumulation_steps = 4
num_epochs =100

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_dice = train_acc = train_iou = train_hd = 0.0
    train_mean_iou = train_precision = train_recall = train_f1 = 0.0

    optimizer.zero_grad()

    for step, (images, masks) in tqdm(enumerate(train_loader)):
        images, masks = images.to(device), masks.to(device)

        with autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)  # [B, C, H, W]
            loss = criterion(outputs, masks) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * accumulation_steps  
        train_dice += dice_score(outputs, masks)
        train_acc += pixel_accuracy(outputs, masks)
        train_iou += iou_score(outputs, masks)
        train_hd += hausdorff_distance(outputs, masks)
        train_mean_iou += iou_score(outputs, masks)  
        train_precision += precision_score(outputs, masks)
        train_recall += recall_score(outputs, masks)
        train_f1 += f1_score(outputs, masks)

    n_train = len(train_loader)
    avg_train_loss = train_loss / n_train
    avg_train_dice = train_dice / n_train
    avg_train_acc = train_acc / n_train
    avg_train_iou = train_iou / n_train
    avg_train_hd = train_hd / n_train
    avg_train_mean_iou = train_mean_iou / n_train
    avg_train_precision = train_precision / n_train
    avg_train_recall = train_recall / n_train
    avg_train_f1 = train_f1 / n_train

    model.eval()
    val_loss = 0.0
    val_dice = val_acc = val_iou = val_hd = 0.0
    val_mean_iou = val_precision = val_recall = val_f1 = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader):
            images, masks = images.to(device), masks.to(device)
            with autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, masks)

            val_loss += loss.item()
            val_dice += dice_score(outputs, masks)
            val_acc += pixel_accuracy(outputs, masks)
            val_iou += iou_score(outputs, masks)
            val_hd += hausdorff_distance(outputs, masks)
            val_mean_iou += iou_score(outputs, masks)
            val_precision += precision_score(outputs, masks)
            val_recall += recall_score(outputs, masks)
            val_f1 += f1_score(outputs, masks)

    n_val = len(val_loader)
    avg_val_loss = val_loss / n_val
    avg_val_dice = val_dice / n_val
    avg_val_acc = val_acc / n_val
    avg_val_iou = val_iou / n_val
    avg_val_hd = val_hd / n_val
    avg_val_mean_iou = val_mean_iou / n_val
    avg_val_precision = val_precision / n_val
    avg_val_recall = val_recall / n_val
    avg_val_f1 = val_f1 / n_val

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch:2d} | "
          f"Train Loss: {avg_train_loss:.6f} | Dice: {avg_train_dice:.4f} | Acc: {avg_train_acc:.4f} | "
          f"IoU: {avg_train_iou:.4f} | mIoU: {avg_train_mean_iou:.4f} | Precision: {avg_train_precision:.4f} | Recall: {avg_train_recall:.4f} | F1: {avg_train_f1:.4f} | HD: {avg_train_hd:.4f} || "
          f"Val Loss: {avg_val_loss:.6f} | Dice: {avg_val_dice:.4f} | Acc: {avg_val_acc:.4f} | "
          f"IoU: {avg_val_iou:.4f} | mIoU: {avg_val_mean_iou:.4f} | Precision: {avg_val_precision:.4f} | Recall: {avg_val_recall:.4f} | F1: {avg_val_f1:.4f} | HD: {avg_val_hd:.4f}")

240it [00:36,  6.61it/s]
100%|██████████| 81/81 [00:06<00:00, 11.78it/s]

Epoch  1 | Train Loss: 0.146827 | Dice: 0.7469 | Acc: 0.9848 | IoU: 0.6671 | mIoU: 0.6671 | Precision: 0.8420 | Recall: 0.7473 | F1: 0.7880 | HD: 5.4810 || Val Loss: 0.155464 | Dice: 0.7321 | Acc: 0.9837 | IoU: 0.6468 | mIoU: 0.6468 | Precision: 0.8495 | Recall: 0.7133 | F1: 0.7733 | HD: 5.8816



240it [00:36,  6.62it/s]
100%|██████████| 81/81 [00:07<00:00, 11.35it/s]

Epoch  2 | Train Loss: 0.139162 | Dice: 0.7607 | Acc: 0.9854 | IoU: 0.6799 | mIoU: 0.6799 | Precision: 0.8442 | Recall: 0.7590 | F1: 0.7966 | HD: 5.2830 || Val Loss: 0.153763 | Dice: 0.7345 | Acc: 0.9841 | IoU: 0.6497 | mIoU: 0.6497 | Precision: 0.8584 | Recall: 0.7144 | F1: 0.7778 | HD: 5.8738



240it [00:36,  6.61it/s]
100%|██████████| 81/81 [00:07<00:00, 11.55it/s]

...

Epoch 98 | Train Loss: 0.052169 | Dice: 0.9121 | Acc: 0.9935 | IoU: 0.8603 | mIoU: 0.8603 | Precision: 0.9302 | Recall: 0.9134 | F1: 0.9209 | HD: 2.4625 || Val Loss: 0.091673 | Dice: 0.8444 | Acc: 0.9896 | IoU: 0.7729 | mIoU: 0.7729 | Precision: 0.9040 | Recall: 0.8327 | F1: 0.8657 | HD: 3.6587



240it [00:36,  6.49it/s]
100%|██████████| 81/81 [00:07<00:00, 11.35it/s]

Epoch 99 | Train Loss: 0.051260 | Dice: 0.9139 | Acc: 0.9935 | IoU: 0.8620 | mIoU: 0.8620 | Precision: 0.9294 | Recall: 0.9148 | F1: 0.9214 | HD: 2.4893 || Val Loss: 0.086567 | Dice: 0.8533 | Acc: 0.9900 | IoU: 0.7832 | mIoU: 0.7832 | Precision: 0.8889 | Recall: 0.8568 | F1: 0.8714 | HD: 3.6574



240it [00:36,  6.50it/s]
100%|██████████| 81/81 [00:07<00:00, 11.11it/s]

Epoch 100 | Train Loss: 0.051441 | Dice: 0.9135 | Acc: 0.9935 | IoU: 0.8631 | mIoU: 0.8631 | Precision: 0.9305 | Recall: 0.9166 | F1: 0.9228 | HD: 2.4046 || Val Loss: 0.087111 | Dice: 0.8525 | Acc: 0.9899 | IoU: 0.7818 | mIoU: 0.7818 | Precision: 0.8950 | Recall: 0.8455 | F1: 0.8685 | HD: 3.6487
